**실행 설정**

입력은 `config/paths.local.json` 또는 `SLC7A5_DATA_ROOT`로 지정합니다. 결과는 이 저장소의 `results/`에 기록합니다. 입력/실행 설명은 저장소 루트 README를 참고하세요. 이 노트북에는 기존 탐색/대체 Figure 셀이 포함되어 있으며 전체 Figure 재현 검증은 별도 단계입니다.

In [ ]:
# Repository paths; inputs are read-only, new files stay in results/.
from pathlib import Path
import os
import sys
_start = Path(os.environ.get("SLC7A5_REPO_ROOT", str(Path.cwd()))).resolve()
_repo = next((p for p in [_start, *_start.parents] if (p / "slc7a5_paths.py").is_file()), None)
if _repo is None:
    raise RuntimeError("Open this notebook inside the repository or set SLC7A5_REPO_ROOT.")
if str(_repo) not in sys.path:
    sys.path.insert(0, str(_repo))
from slc7a5_paths import artifact_path, dataset_path, input_path, output_path, output_dir, setup_notebook
setup_notebook('figure1_nucseq')


In [ ]:
# Import package
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scanpy as sc
import seaborn as sns
import matplotlib.colors as mcolors
import scipy.sparse as sp



In [ ]:
import matplotlib as mpl

mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams["svg.fonttype"] = "none"
mpl.rcParams["pdf.use14corefonts"] = False

In [ ]:
# adata 경로를 실제 파일 경로로 수정하세요
adata = sc.read(artifact_path('50_Public_GEO_analysis/analysis/Data/Liver_DB_nuc_stage.h5ad'))
print(f"전체 세포 수: {adata.n_obs:,}, 유전자 수: {adata.n_vars:,}")
adata

In [ ]:
# adata 경로를 실제 파일 경로로 수정하세요
adata1 = sc.read(artifact_path('external/MASLD_NUC_DB_total.h5ad'))
print(f"전체 세포 수: {adata.n_obs:,}, 유전자 수: {adata.n_vars:,}")
adata1

In [ ]:
adata_GSE210077=sc.read(artifact_path('50_Public_GEO_analysis/analysis/01_MergeNucseq/GSE210077_Raw.h5ad'))
adata_GSE210077

In [ ]:
adata.obs['journal'].value_counts()

In [ ]:
adata.obs['journal'] = adata.obs['journal'].replace({
    'GSE210077': 'TEMP',
    'GSE212837': 'GSE210077',
    'TEMP': 'GSE212837'
})

In [ ]:
adata.obs['journal'].value_counts()

In [ ]:
adata.obs['journal'] = adata.obs['journal'].replace({
    'TEMP': 'GSE212837'
})
adata.obs['journal'].value_counts()

In [ ]:
adata1.obs['journal'] = adata1.obs['journal'].replace({
    'GSE210077': 'TEMP',
    'GSE212837': 'GSE210077',
    'TEMP': 'GSE212837'
})
adata1.obs['journal'] = adata1.obs['journal'].replace({
    'TEMP': 'GSE212837'
})
adata1.obs['journal'].value_counts()

In [ ]:
# adata 경로를 실제 파일 경로로 수정하세요
adata.write(output_path('50_Public_GEO_analysis/analysis/Data/Liver_DB_nuc_stage.h5ad'))
adata1.write(output_path('external/MASLD_NUC_DB_total.h5ad'))

In [ ]:
sc.pl.umap(adata, color=['celltype'])

In [ ]:
sc.pl.umap(adata, color=['Disease_status'])

# Nucseq_analysis MASLD vs Healthy

In [ ]:
adata.obs["disease_group"] = adata.obs["Disease_status"].astype(str).copy()
adata.obs["disease_group"] = adata.obs["disease_group"].replace({
    "MASL": "MASLD",
    "MASH": "MASLD",
})

In [ ]:
target_groups = ["Healthy", "MASLD"]
adata_deg = adata[adata.obs["disease_group"].isin(target_groups)].copy()

In [ ]:
adata_deg.obs["disease_group"] = pd.Categorical(
    adata_deg.obs["disease_group"],
    categories=["Healthy", "MASLD"]
)

In [ ]:
adata_deg

In [ ]:
sc.tl.rank_genes_groups(
    adata_deg,
    groupby='disease_group',
    method='wilcoxon',
    use_raw=True,
    pts=True
)

sc.pl.rank_genes_groups_dotplot(adata_deg, n_genes=5)
deg_all = sc.get.rank_genes_groups_df(adata_deg, group=None)
deg_all

In [ ]:
sc.pl.rank_genes_groups_dotplot(adata_deg, n_genes=5)
deg_all = sc.get.rank_genes_groups_df(adata_deg, group=None)
deg_all

In [ ]:
deg_all[deg_all["names"] == "SLC7A5"]

In [ ]:
top_genes = deg_all.groupby("group")["names"].apply(lambda x: x.head(20)).explode().unique().tolist()
sc.pl.heatmap(
    adata_deg,
    var_names=top_genes,
    groupby="disease_group",
    use_raw=False,
    cmap="bwr",
    vmin=-3,
    vmax=3,
    swap_axes=True,
    dendrogram=False,
    show=False, 
    #return_fig=True,
    figsize=(12, 7)
)

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
from scipy import sparse

genes_wanted = [
    'NBEAL1', 'TMSB4X', 'UBA52', 'IGKC', 'CCL3', 'TOMM7', 'ZNF90', 'ID2', 'CCL4',
    'MALAT1', 'MTRNR2L8', 'MYL12A', 'COX7C', 'JUNB', 'SERF2', 'TXNIP', 'FOS',
    'TMSB10', 'TPT1', 'PFDN5', 'JUND', 'METRNL', 'EEF1G', 'TGFB1', 'ATP6V0C',
    'PTPRCAP', 'EZR', 'TNFAIP3', 'GNAS', 'SLC7A5', 'CREM', 'RBM38', 'RNASEK',
    'CD81', 'HSPA5', 'FAM177A1', 'EMD', 'TUBA4A', 'VPS37B', 'TNFRSF1B'
]

existing_genes = [g for g in genes_wanted if g in adata_deg.var_names]
missing_genes = [g for g in genes_wanted if g not in adata_deg.var_names]

print("Existing genes:", len(existing_genes))
print("Missing genes:", missing_genes)

# 있는 유전자만 먼저 subset
adata_exist = adata_deg[:, existing_genes].copy()

X_exist = adata_exist.X
if not sparse.issparse(X_exist):
    X_exist = sparse.csr_matrix(X_exist)

# 없는 유전자 수만큼 0 matrix 생성
n_cells = adata_exist.n_obs
X_missing = sparse.csr_matrix((n_cells, len(missing_genes)), dtype=X_exist.dtype)

# X를 직접 붙이기
X_new = sparse.hstack([X_exist, X_missing], format="csr")

# 새 var 만들기
var_new = pd.DataFrame(index=existing_genes + missing_genes)

# varm/obsm/layers 꼬임 없이 새 AnnData 생성
adata_plot = sc.AnnData(
    X=X_new,
    obs=adata_exist.obs.copy(),
    var=var_new
)

# 원래 원하는 순서로 재정렬
adata_plot = adata_plot[:, genes_wanted].copy()

sc.pl.heatmap(
    adata_plot,
    var_names=genes_wanted,
    groupby="disease_group",
    use_raw=False,
    cmap="bwr",
    vmin=-3,
    vmax=3,
    swap_axes=True,
    dendrogram=False,
    show=False,
    figsize=(12, 7)
)

plt.savefig(
    output_path('02_SLC7A5_mono_Journal/05_Figure/nucseq_heatmap_Fig1Cv2.pdf'),
    bbox_inches="tight",
    dpi=300
)
plt.close()

In [ ]:
deg_all.to_csv(output_path('02_SLC7A5_mono_Journal/05_Figure/snRNA_state_DEG.csv'))

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import sparse

sample_col = "sample"   # 실제 컬럼명으로 바꾸세요

deg_all = sc.get.rank_genes_groups_df(adata_deg, group=None)

exclude_prefix = ("RPS", "RPL", "MT-")
exclude_exact = {"XIST", "FOS", "FOSB", "JUN", "JUNB"}

deg_all = deg_all[
    (deg_all["pvals_adj"] < 0.05) &
    (deg_all["logfoldchanges"].abs() > 0.25)
].copy()

deg_all = deg_all[
    ~deg_all["names"].str.startswith(exclude_prefix, na=False)
].copy()

deg_all = deg_all[
    ~deg_all["names"].isin(exclude_exact)
].copy()

top_genes = (
    deg_all.groupby("group", sort=False)["names"]
    .apply(lambda x: x.drop_duplicates().head(20))
    .explode()
    .dropna()
    .unique()
    .tolist()
)

adata_sub = adata_deg[:, top_genes].copy()

X = adata_sub.X
if sparse.issparse(X):
    X = X.toarray()

expr = pd.DataFrame(X, index=adata_sub.obs_names, columns=adata_sub.var_names)
meta = adata_sub.obs[[sample_col, "disease_group"]].copy()

expr["sample"] = meta[sample_col].values
expr_mean = expr.groupby("sample").mean()

sample_meta = (
    meta.reset_index()
    .groupby(sample_col)["disease_group"]
    .agg(lambda x: x.iloc[0])
)

# gene-wise z-score
expr_mean_z = expr_mean.sub(expr_mean.mean(axis=0), axis=1)
expr_mean_z = expr_mean_z.div(expr_mean.std(axis=0).replace(0, np.nan), axis=1)
expr_mean_z = expr_mean_z.fillna(0)

# sample annotation color
group_palette = {"Healthy": "#4C9BE8", "MASLD": "#F39C34"}
col_colors = sample_meta.map(group_palette)

g = sns.clustermap(
    expr_mean_z.T,
    cmap="RdBu_r",
    center=0,
    col_colors=col_colors,
    linewidths=0,
    figsize=(12, 8),
    xticklabels=True,
    yticklabels=True,
)

plt.show()

In [ ]:
# RPS, RPL, MT 제외 필터
top_genes = (
    deg_all
    .groupby("group")["names"]
    .apply(lambda x: x.head(100))  # 넉넉하게 뽑아서 필터 후 20개 확보
    .explode()
    .reset_index(drop=True)
)

filtered_genes = top_genes[
    ~top_genes.str.startswith(("RPS", "RPL", "MT-"))
]

top_genes_final = (
    deg_all[deg_all["names"].isin(filtered_genes)]
    .groupby("group")["names"]
    .apply(lambda x: x.head(20))
    .explode()
    .unique()
    .tolist()
)

sc.pl.heatmap(
    adata_deg,
    var_names=top_genes_final,
    groupby="disease_group",
    use_raw=False,
    cmap="bwr",
    vmin=-3,
    vmax=3,
    swap_axes=True,
    dendrogram=False,
    show=True,
    figsize=(12, 7)
)

#plt.savefig("/home/jaehyunchoi/Liver_bio_project/02_SLC7A5_mono_Journal/05_Figure/cellseq_heatmap_Fig1B.pdf", bbox_inches='tight', dpi=300)
plt.close()

In [ ]:
deg_all = sc.get.rank_genes_groups_df(adata_deg, group=None)

# 필요하면 유의성/FC 필터 먼저
deg_all = deg_all[
    (deg_all["pvals_adj"] < 0.05) &
    (deg_all["logfoldchanges"].abs() > 0.25)
].copy()

# 제외할 prefix
exclude_prefix = ("RPS", "RPL", "MT-")

# group별로 top gene 선정
top_n = 15
top_genes_by_group = {}

for grp, df_grp in deg_all.groupby("group", sort=False):
    df_grp = df_grp.loc[
        ~df_grp["names"].str.startswith(exclude_prefix, na=False)
    ].copy()

    # group marker 기준: 해당 group에서 up인 것만 보고 싶으면 logFC > 0
    df_grp = df_grp[df_grp["logfoldchanges"] > 0].copy()

    # p값 우선 정렬
    df_grp = df_grp.sort_values(["pvals_adj", "logfoldchanges"], ascending=[True, False])

    top_genes_by_group[grp] = df_grp["names"].head(top_n).tolist()

# 순서 보존하며 합치기
top_genes_final = []
for grp in top_genes_by_group:
    for g in top_genes_by_group[grp]:
        if g not in top_genes_final:
            top_genes_final.append(g)

print(top_genes_by_group)
print(top_genes_final)

sc.pl.heatmap(
    adata_deg,
    var_names=top_genes_final,
    groupby="disease_group",
    use_raw=False,
    cmap="bwr",
    standard_scale="var",   # 중요
    swap_axes=True,
    dendrogram=False,
    show=True,
    figsize=(10, 8),
)

plt.close()

In [ ]:
import scanpy as sc

deg_all = sc.get.rank_genes_groups_df(adata_deg, group=None)

exclude_prefix = ("RPS", "RPL", "MT-")
exclude_exact = {"XIST", "FOS", "FOSB", "JUN", "JUNB"}

deg_all = deg_all[
    (deg_all["pvals_adj"] < 0.05) &
    (deg_all["logfoldchanges"] > 0.25)
].copy()

deg_all = deg_all[
    ~deg_all["names"].str.startswith(exclude_prefix, na=False)
].copy()

deg_all = deg_all[
    ~deg_all["names"].isin(exclude_exact)
].copy()

top_n = 20
var_dict = {}

for grp, df_grp in deg_all.groupby("group", sort=False):
    genes = (
        df_grp["names"]
        .drop_duplicates()
        .head(top_n)
        .tolist()
    )
    var_dict[grp] = genes

sc.pl.matrixplot(
    adata_deg,
    var_names=var_dict,
    groupby="disease_group",
    use_raw=False,
    cmap="RdBu_r",
    standard_scale="var",
    dendrogram=False,
    figsize=(8, 5),
)

In [ ]:
adata

In [ ]:
adata_deg.obs['disease_group'].value_counts()

In [ ]:
import numpy as np

import scipy.sparse as sp

x = adata_deg.X

if sp.issparse(x):

    arr = x.data

else:

    arr = np.ravel(x)

print("min:", arr.min())

print("max:", arr.max())

print("mean:", arr.mean())

print("example unique:", np.unique(arr[:10000])[:20])

In [ ]:
summary_rows = []
for celltype in desired_celltypes:
    sub = plot_df.loc[plot_df["celltype"] == celltype].copy()

    healthy = sub.loc[sub["disease_group"] == "Healthy", "expression"].values
    masld = sub.loc[sub["disease_group"] == "MASLD", "expression"].values

    healthy_nonzero = healthy[healthy > 0]
    masld_nonzero = masld[masld > 0]

    summary_rows.append({
        "celltype": celltype,
        "n_healthy": len(healthy),
        "n_masld": len(masld),
        "healthy_pct_expr": (healthy > 0).mean() * 100 if len(healthy) > 0 else np.nan,
        "masld_pct_expr": (masld > 0).mean() * 100 if len(masld) > 0 else np.nan,
        "healthy_mean": np.mean(healthy) if len(healthy) > 0 else np.nan,
        "masld_mean": np.mean(masld) if len(masld) > 0 else np.nan,
        "healthy_nonzero_median": np.median(healthy_nonzero) if len(healthy_nonzero) > 0 else np.nan,
        "masld_nonzero_median": np.median(masld_nonzero) if len(masld_nonzero) > 0 else np.nan,
    })

summary_df = pd.DataFrame(summary_rows)
print(summary_df.round(4).to_string(index=False))

In [ ]:
test_results = []
for celltype in desired_celltypes:
    sub = plot_df.loc[plot_df["celltype"] == celltype].copy()
    healthy = sub.loc[sub["disease_group"] == "Healthy", "expression"].values
    masld = sub.loc[sub["disease_group"] == "MASLD", "expression"].values

    healthy_pct = (healthy > 0).mean() if len(healthy) > 0 else np.nan
    masld_pct = (masld > 0).mean() if len(masld) > 0 else np.nan

    if np.isnan(healthy_pct) or np.isnan(masld_pct):
        higher_group = "NA"
    elif healthy_pct > masld_pct:
        higher_group = "Healthy"
    elif masld_pct > healthy_pct:
        higher_group = "MASLD"
    else:
        higher_group = "Equal"

    if len(healthy) == 0 or len(masld) == 0:
        pvalue = np.nan
    else:
        _, pvalue = mannwhitneyu(healthy, masld, alternative="two-sided")

    test_results.append({
        "celltype": celltype,
        "n_healthy": len(healthy),
        "n_masld": len(masld),
        "healthy_pct_expr": healthy_pct * 100 if not np.isnan(healthy_pct) else np.nan,
        "masld_pct_expr": masld_pct * 100 if not np.isnan(masld_pct) else np.nan,
        "higher_group": higher_group,
        "pvalue": pvalue,
    })

In [ ]:
# file: violin_slc7a5_by_celltype_disease_with_pctexpr_direction.py

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import scipy.sparse as sp

from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

gene = "SLC7A5"
celltype_col = "celltype"
condition_col = "disease_group"

desired_celltypes = [
    "Hepatocyte",
    "Cholangiocyte",
    "Lymphocyte",
    "Macrophage",
    "B cells",
    "Stellete",
    "Endothelial",
]
desired_conditions = ["Healthy", "MASLD"]

mask = (
    adata_deg.obs[celltype_col].isin(desired_celltypes)
    & adata_deg.obs[condition_col].isin(desired_conditions)
)

# Why: scaled X가 아니라 raw/log-normalized 값을 써야 해석이 맞음
expr_source = adata_deg.raw if adata_deg.raw is not None else adata_deg

if gene not in expr_source.var_names:
    raise ValueError(f"{gene} not found in expression source var_names")

gene_idx = expr_source.var_names.get_loc(gene)
expr = expr_source.X[mask, gene_idx]

if sp.issparse(expr):
    expr = expr.toarray().ravel()
else:
    expr = np.asarray(expr).ravel()

plot_df = adata_deg.obs.loc[mask, [celltype_col, condition_col]].copy()
plot_df.columns = ["celltype", "disease_group"]
plot_df["expression"] = expr

plot_df["celltype"] = pd.Categorical(
    plot_df["celltype"],
    categories=desired_celltypes,
    ordered=True,
)
plot_df["disease_group"] = pd.Categorical(
    plot_df["disease_group"],
    categories=desired_conditions,
    ordered=True,
)

plot_df = plot_df.dropna(subset=["celltype", "disease_group", "expression"]).copy()

def p_to_star(p: float) -> str:
    if pd.isna(p):
        return "NA"
    if p < 0.001:
        return "***"
    if p < 0.01:
        return "**"
    if p < 0.05:
        return "*"
    return "ns"

def direction_from_pct(healthy_pct: float, masld_pct: float) -> str:
    if pd.isna(healthy_pct) or pd.isna(masld_pct):
        return "NA"
    if healthy_pct > masld_pct:
        return "↑ H"
    if masld_pct > healthy_pct:
        return "↑ M"
    return "="

summary_rows = []
for celltype in desired_celltypes:
    sub = plot_df.loc[plot_df["celltype"] == celltype].copy()
    healthy = sub.loc[sub["disease_group"] == "Healthy", "expression"].values
    masld = sub.loc[sub["disease_group"] == "MASLD", "expression"].values

    healthy_nonzero = healthy[healthy > 0]
    masld_nonzero = masld[masld > 0]

    healthy_pct_expr = (healthy > 0).mean() * 100 if len(healthy) > 0 else np.nan
    masld_pct_expr = (masld > 0).mean() * 100 if len(masld) > 0 else np.nan

    healthy_mean_expr = np.mean(healthy) if len(healthy) > 0 else np.nan
    masld_mean_expr = np.mean(masld) if len(masld) > 0 else np.nan

    healthy_nonzero_mean = np.mean(healthy_nonzero) if len(healthy_nonzero) > 0 else np.nan
    masld_nonzero_mean = np.mean(masld_nonzero) if len(masld_nonzero) > 0 else np.nan

    healthy_median = np.median(healthy) if len(healthy) > 0 else np.nan
    masld_median = np.median(masld) if len(masld) > 0 else np.nan

    if len(healthy) == 0 or len(masld) == 0:
        pvalue = np.nan
    else:
        _, pvalue = mannwhitneyu(healthy, masld, alternative="two-sided")

    direction_label = direction_from_pct(healthy_pct_expr, masld_pct_expr)

    summary_rows.append(
        {
            "celltype": celltype,
            "n_healthy": len(healthy),
            "n_masld": len(masld),
            "healthy_pct_expr": healthy_pct_expr,
            "masld_pct_expr": masld_pct_expr,
            "healthy_mean_expr": healthy_mean_expr,
            "masld_mean_expr": masld_mean_expr,
            "healthy_nonzero_mean": healthy_nonzero_mean,
            "masld_nonzero_mean": masld_nonzero_mean,
            "healthy_median": healthy_median,
            "masld_median": masld_median,
            "direction_label": direction_label,
            "pvalue": pvalue,
        }
    )

stats_df = pd.DataFrame(summary_rows)

valid_mask = stats_df["pvalue"].notna()
stats_df["p_adj"] = np.nan
if valid_mask.sum() > 0:
    _, p_adj, _, _ = multipletests(
        stats_df.loc[valid_mask, "pvalue"].values,
        alpha=0.05,
        method="fdr_bh",
    )
    stats_df.loc[valid_mask, "p_adj"] = p_adj

stats_df["star"] = stats_df["p_adj"].apply(p_to_star)

display_cols = [
    "celltype",
    "n_healthy",
    "n_masld",
    "healthy_pct_expr",
    "masld_pct_expr",
    "healthy_mean_expr",
    "masld_mean_expr",
    "healthy_nonzero_mean",
    "masld_nonzero_mean",
    "healthy_median",
    "masld_median",
    "direction_label",
    "pvalue",
    "p_adj",
    "star",
]

print(stats_df[display_cols].round(4).to_string(index=False))

#plt.figure(figsize=(max(10, len(desired_celltypes) * 1.6), 6.8))
plt.figure(figsize=(14, 5))
ax = sns.violinplot(
    data=plot_df,
    x="celltype",
    y="expression",
    hue="disease_group",
    order=desired_celltypes,
    hue_order=desired_conditions,
    dodge=True,
    cut=0,
    inner=None,
    linewidth=1,
    palette={"Healthy": "#d62728", "MASLD": "#1f77b4"},
)

plot_df_sampled = (
    plot_df.groupby(["celltype", "disease_group"], observed=True, group_keys=False)
    .apply(lambda x: x.sample(min(len(x), 1000), random_state=42))
    .reset_index(drop=True)
)

sns.stripplot(
    data=plot_df_sampled,
    x="celltype",
    y="expression",
    hue="disease_group",
    order=desired_celltypes,
    hue_order=desired_conditions,
    dodge=True,
    jitter=0.16,
    size=1.0,
    alpha=0.20,
    palette={"Healthy": "#d62728", "MASLD": "#1f77b4"},
    ax=ax,
)

median_df = (
    plot_df.groupby(["celltype", "disease_group"], observed=True)["expression"]
    .median()
    .reset_index()
)

position_map = {"Healthy": -0.2, "MASLD": 0.2}
color_map = {"Healthy": "#8b1e1e", "MASLD": "#0f4c81"}

for _, row in median_df.iterrows():
    x_center = desired_celltypes.index(row["celltype"])
    x = x_center + position_map[row["disease_group"]]
    ax.scatter(
        x,
        row["expression"],
        s=45,
        color=color_map[row["disease_group"]],
        edgecolor="black",
        zorder=5,
    )

y_min = float(plot_df["expression"].min())
y_max = float(plot_df["expression"].max())
y_range = y_max - y_min if y_max > y_min else 1.0
ax.set_ylim(y_min, y_max + y_range * 0.42)

for i, celltype in enumerate(desired_celltypes):
    sub = plot_df.loc[plot_df["celltype"] == celltype, "expression"]
    if sub.empty:
        continue

    row = stats_df.loc[stats_df["celltype"] == celltype]
    if row.empty:
        continue

    star = row["star"].iloc[0]
    direction_label = row["direction_label"].iloc[0]
    healthy_pct = row["healthy_pct_expr"].iloc[0]
    masld_pct = row["masld_pct_expr"].iloc[0]
    healthy_mean = row["healthy_mean_expr"].iloc[0]
    masld_mean = row["masld_mean_expr"].iloc[0]

    y = sub.max() + y_range * 0.07
    x1 = i - 0.2
    x2 = i + 0.2

    ax.plot(
        [x1, x1, x2, x2],
        [y, y + y_range * 0.02, y + y_range * 0.02, y],
        lw=1,
        c="black",
    )
    ax.text(
        i,
        y + y_range * 0.025,
        f"{star} {direction_label}",
        ha="center",
        va="bottom",
        fontsize=10,
        fontweight="bold",
    )
    ax.text(
        i,
        y + y_range * 0.09,
        f"H {healthy_pct:.1f}% / M {masld_pct:.1f}%",
        ha="center",
        va="bottom",
        fontsize=8,
        color="#444444",
    )
    ax.text(
        i,
        y + y_range * 0.14,
        f"mean: {healthy_mean:.2f} / {masld_mean:.2f}",
        ha="center",
        va="bottom",
        fontsize=8,
        color="#444444",
    )

handles, labels = ax.get_legend_handles_labels()
legend_map = {}
for h, l in zip(handles, labels):
    if l not in legend_map:
        legend_map[l] = h

ax.legend(
    [legend_map["Healthy"], legend_map["MASLD"]],
    ["Healthy", "MASLD"],
    title="",
    frameon=False,
    loc="upper right",
)

ax.set_xlabel("")
ax.set_ylabel(f"{gene} expression")
ax.set_title(
    f"{gene} expression by cell type and disease group\n"
    f"Direction uses % expressing (>0), stars use BH-adjusted Mann–Whitney p-values"
)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig(output_path('02_SLC7A5_mono_Journal/05_Figure/Fig1D_Nucseq.pdf'), bbox_inches='tight')
plt.show()

In [ ]:
sc.pl.umap(adata, color=['celltype'])

In [ ]:
adata_mac = adata[adata.obs["celltype"].isin(["Macrophage"])].copy()

In [ ]:
adata_mac

# Figure Nucseq pos neg

In [ ]:
# Fig1E nucseq version: SLC7A5 top/bottom 20% DEG and focused GSEA heatmap
import re
from pathlib import Path

import numpy as np
import pandas as pd
import scanpy as sc
import gseapy as gp
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from scipy import sparse

H5AD_PATH = artifact_path('50_Public_GEO_analysis/analysis/Data/Liver_DB_nuc_stage.h5ad')
FIG_DIR = Path(output_dir('02_SLC7A5_mono_Journal/05_Figure'))
OUTDIR = Path(output_dir('02_SLC7A5_mono_Journal/02_Analysis/Nucseq/Fig1E_Nucseq_SLC7A5_top20_bottom20_DEG_GSEA'))
FIG_DIR.mkdir(parents=True, exist_ok=True)
OUTDIR.mkdir(parents=True, exist_ok=True)

GENE = "SLC7A5"
TOP_BOTTOM_PCT = 0.20
MIN_CELLS_PER_GROUP = 25
LFC_CAP = 10.0
TARGET_CELLTYPES = ["Hepatocyte", "Stellete", "Macrophage"]
GROUP_COL = "SLC7A5_top20_bottom20_group"

# Use the currently loaded nucseq adata when available; otherwise read it from disk.
if "adata" in globals():
    adata_fig1e_nuc = adata.copy()
else:
    adata_fig1e_nuc = sc.read_h5ad(H5AD_PATH)

print(adata_fig1e_nuc)
print("raw exists:", adata_fig1e_nuc.raw is not None)
print("celltype counts:")
display(adata_fig1e_nuc.obs["celltype"].value_counts())

def get_gene_vector(adata_obj, gene):
    gene_upper = gene.upper()
    if adata_obj.raw is not None:
        raw_var = pd.Index([str(g).upper() for g in adata_obj.raw.var_names])
        if gene_upper in raw_var:
            raw_gene = adata_obj.raw.var_names[raw_var.get_loc(gene_upper)]
            x = adata_obj.raw[:, raw_gene].X
            return np.asarray(x.toarray()).ravel() if sparse.issparse(x) else np.asarray(x).ravel()

    var = pd.Index([str(g).upper() for g in adata_obj.var_names])
    if gene_upper not in var:
        raise ValueError(f"{gene} not found in adata.var_names or adata.raw.var_names")
    x = adata_obj[:, adata_obj.var_names[var.get_loc(gene_upper)]].X
    return np.asarray(x.toarray()).ravel() if sparse.issparse(x) else np.asarray(x).ravel()

def safe_name(value):
    return re.sub(r"[^A-Za-z0-9]+", "_", str(value)).strip("_")

adata_fig1e_nuc.obs[f"{GENE}_expr_raw"] = get_gene_vector(adata_fig1e_nuc, GENE)

available_celltypes = set(adata_fig1e_nuc.obs["celltype"].astype(str).unique())
missing = [ct for ct in TARGET_CELLTYPES if ct not in available_celltypes]
if missing:
    print("Missing target celltype labels:", missing)
    print("Available labels:", sorted(available_celltypes))

adata_fig1e_nuc_sub = adata_fig1e_nuc[adata_fig1e_nuc.obs["celltype"].isin(TARGET_CELLTYPES)].copy()
print("Selected celltype counts:")
display(adata_fig1e_nuc_sub.obs["celltype"].value_counts())

# DEG: SLC7A5-high top 20% vs SLC7A5-low bottom 20% per nucseq cell type.
deg_results = {}
summary_rows = []
use_raw_for_deg = adata_fig1e_nuc_sub.raw is not None

for ct in TARGET_CELLTYPES:
    sub = adata_fig1e_nuc_sub[adata_fig1e_nuc_sub.obs["celltype"] == ct].copy()
    n_total = sub.n_obs
    n_group = int(np.floor(n_total * TOP_BOTTOM_PCT))

    if n_group < MIN_CELLS_PER_GROUP:
        print(f"[{ct}] skip: n_total={n_total}, n_group={n_group} < {MIN_CELLS_PER_GROUP}")
        continue

    expr = sub.obs[f"{GENE}_expr_raw"].to_numpy()
    order = np.argsort(expr, kind="mergesort")

    group = np.full(n_total, "middle", dtype=object)
    group[order[:n_group]] = "SLC7A5_low"
    group[order[-n_group:]] = "SLC7A5_high"
    sub.obs[GROUP_COL] = pd.Categorical(
        group,
        categories=["SLC7A5_low", "middle", "SLC7A5_high"],
        ordered=True,
    )

    sub_tb = sub[sub.obs[GROUP_COL] != "middle"].copy()
    sub_tb.obs[GROUP_COL] = sub_tb.obs[GROUP_COL].cat.remove_unused_categories()

    sc.tl.rank_genes_groups(
        sub_tb,
        groupby=GROUP_COL,
        groups=["SLC7A5_high"],
        reference="SLC7A5_low",
        method="wilcoxon",
        use_raw=use_raw_for_deg,
        pts=True,
    )

    deg = sc.get.rank_genes_groups_df(sub_tb, group="SLC7A5_high").rename(columns={"names": "gene"})
    deg["logfoldchanges"] = pd.to_numeric(deg["logfoldchanges"], errors="coerce").clip(-LFC_CAP, LFC_CAP)
    deg["scores"] = pd.to_numeric(deg["scores"], errors="coerce")
    deg["pvals"] = pd.to_numeric(deg["pvals"], errors="coerce")
    deg["pvals_adj"] = pd.to_numeric(deg["pvals_adj"], errors="coerce")
    deg.insert(0, "celltype", ct)

    deg_path = OUTDIR / f"DEG_{safe_name(ct)}_SLC7A5_top20_high_vs_bottom20_low.csv"
    deg.to_csv(deg_path, index=False)
    deg_results[ct] = deg

    summary_rows.append({
        "celltype": ct,
        "total_cells": n_total,
        "n_high": int((sub_tb.obs[GROUP_COL] == "SLC7A5_high").sum()),
        "n_low": int((sub_tb.obs[GROUP_COL] == "SLC7A5_low").sum()),
        "low_expr_max": float(expr[order[:n_group]].max()),
        "high_expr_min": float(expr[order[-n_group:]].min()),
        "deg_csv": str(deg_path),
        "logfc_na": int(deg["logfoldchanges"].isna().sum()),
        "sig_deg_fdr_0_05": int((deg["pvals_adj"] < 0.05).sum()),
    })

    print(f"[{ct}] DEG saved: {deg_path.name} | n/group={n_group} | logFC NaN={deg['logfoldchanges'].isna().sum()}")

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(OUTDIR / "Fig1E_Nucseq_SLC7A5_top20_bottom20_DEG_summary.csv", index=False)
display(summary_df)

# Focused GSEA for Glycolysis, TCA cycle, OXPHOS, FAO, mTOR, and NF-kB-related pathways.
GENE_SET_COLLECTIONS = {
    "hallmark": "MSigDB_Hallmark_2020",
    "kegg": "KEGG_2021_Human",
}
FOCUS_PATTERNS = {
    "Glycolysis": r"GLYCOLYSIS|GLUCONEOGENESIS|GLUCOSE",
    "TCA cycle": r"TCA|CITRATE_CYCLE|TRICARBOXYLIC",
    "OXPHOS": r"OXIDATIVE_PHOSPHORYLATION|OXPHOS|RESPIRATORY_CHAIN|ELECTRON_TRANSPORT_CHAIN",
    "FAO": r"FATTY_ACID_METABOLISM|FATTY_ACID_BETA_OXIDATION|BETA_OXIDATION|FATTY_ACID_DEGRADATION|PPAR",
    "mTOR": r"MTOR|MTORC1|PI3K_AKT_MTOR|AKT_MTOR",
    "NF-kB": r"NFKB|NF_KB|TNFA_SIGNALING_VIA_NFKB|TOLL_LIKE|INFLAMMATORY_RESPONSE",
}

def make_prerank(deg):
    rank_df = deg[["gene", "logfoldchanges", "pvals", "scores"]].copy()
    rank_df["gene"] = rank_df["gene"].astype(str).str.upper()
    rank_df["pvals"] = pd.to_numeric(rank_df["pvals"], errors="coerce").clip(lower=1e-300)
    rank_df["logfoldchanges"] = pd.to_numeric(rank_df["logfoldchanges"], errors="coerce")
    rank_df["scores"] = pd.to_numeric(rank_df["scores"], errors="coerce")
    rank_df["rank_metric"] = np.sign(rank_df["logfoldchanges"]) * -np.log10(rank_df["pvals"])
    rank_df.loc[rank_df["rank_metric"].isna(), "rank_metric"] = rank_df.loc[rank_df["rank_metric"].isna(), "scores"]
    rank_df = rank_df.replace([np.inf, -np.inf], np.nan).dropna(subset=["gene", "rank_metric"])
    rank_df = rank_df.sort_values("rank_metric", ascending=False).drop_duplicates("gene", keep="first")
    return rank_df[["gene", "rank_metric"]]

gsea_results = {}
focused_rows = []

for ct, deg in deg_results.items():
    prerank = make_prerank(deg)
    prerank_path = OUTDIR / f"Prerank_{safe_name(ct)}_SLC7A5_top20_high_vs_low.rnk"
    prerank.to_csv(prerank_path, sep="\t", index=False, header=False)

    for label, gene_sets in GENE_SET_COLLECTIONS.items():
        pre_res = gp.prerank(
            rnk=prerank,
            gene_sets=gene_sets,
            min_size=5,
            max_size=500,
            permutation_num=1000,
            seed=7,
            outdir=None,
            verbose=False,
        )

        res = pre_res.res2d.copy()
        res.insert(0, "celltype", ct)
        res.insert(1, "gene_set", label)
        res_path = OUTDIR / f"GSEA_{safe_name(ct)}_{label}_SLC7A5_top20_high_vs_low.csv"
        res.to_csv(res_path, index=False)
        gsea_results[(ct, label)] = res

        term_norm = res["Term"].astype(str).str.upper().str.replace(" ", "_", regex=False).str.replace("-", "_", regex=False)
        for pathway_group, pattern in FOCUS_PATTERNS.items():
            focus = res.loc[term_norm.str.contains(pattern, regex=True, na=False)].copy()
            if focus.empty:
                continue
            focus.insert(2, "pathway_group", pathway_group)
            focused_rows.append(focus)

        n_focus = sum(term_norm.str.contains(pattern, regex=True, na=False).sum() for pattern in FOCUS_PATTERNS.values())
        print(f"[{ct} | {label}] GSEA saved: {res_path.name} | focused matches={n_focus}")

focused_gsea_df = pd.concat(focused_rows, ignore_index=True) if focused_rows else pd.DataFrame()
focused_gsea_df.to_csv(OUTDIR / "GSEA_focused_glycolysis_TCA_OXPHOS_FAO_mTOR_NFKB_all_matches.csv", index=False)
display(focused_gsea_df.head(30))

# Fig1E nucseq focused NES heatmap: one strongest matched term per pathway group and cell type.
if focused_gsea_df.empty:
    raise ValueError("No focused GSEA terms matched Glycolysis, TCA cycle, OXPHOS, FAO, mTOR, or NF-kB patterns.")

plot_rows = []
for (ct, pathway_group), sub in focused_gsea_df.groupby(["celltype", "pathway_group"], observed=True):
    sub = sub.copy()
    sub["abs_nes"] = pd.to_numeric(sub["NES"], errors="coerce").abs()
    best = sub.sort_values(["abs_nes", "FDR q-val"], ascending=[False, True]).iloc[0]
    plot_rows.append(best)

plot_df = pd.DataFrame(plot_rows)
plot_df.to_csv(OUTDIR / "GSEA_focused_glycolysis_TCA_OXPHOS_FAO_mTOR_NFKB_best_terms_for_heatmap.csv", index=False)

pathway_order = ["Glycolysis", "TCA cycle", "OXPHOS", "FAO", "mTOR", "NF-kB"]
celltype_order = [ct for ct in TARGET_CELLTYPES if ct in plot_df["celltype"].unique()]
nes_mat = pd.DataFrame(np.nan, index=pathway_order, columns=celltype_order)
label_mat = pd.DataFrame("", index=pathway_order, columns=celltype_order)

for _, row in plot_df.iterrows():
    ct = row["celltype"]
    pg = row["pathway_group"]
    if pg not in nes_mat.index or ct not in nes_mat.columns:
        continue
    nes_mat.loc[pg, ct] = float(row["NES"])
    p = float(row["NOM p-val"])
    fdr = float(row["FDR q-val"])
    star = "**" if p < 0.01 else "*" if p < 0.05 else ""
    label_mat.loc[pg, ct] = f"{star}\n{row['gene_set']}: {row['Term']}\nFDR={fdr:.3g}"

vmax = np.nanmax(np.abs(nes_mat.to_numpy(dtype=float)))
vmax = 1 if not np.isfinite(vmax) or vmax == 0 else max(1, vmax)

fig, ax = plt.subplots(figsize=(max(5.5, 1.8 * len(celltype_order)), 5.8))
im = ax.imshow(
    nes_mat.to_numpy(dtype=float),
    cmap="RdBu_r",
    norm=mcolors.TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax),
    aspect="auto",
)

ax.set_xticks(range(len(celltype_order)))
ax.set_xticklabels(celltype_order, rotation=35, ha="right")
ax.set_yticks(range(len(pathway_order)))
ax.set_yticklabels(pathway_order)
ax.set_title("Fig1E Nucseq. Focused GSEA: SLC7A5 top 20% high vs bottom 20% low")

for i, pathway in enumerate(pathway_order):
    for j, ct in enumerate(celltype_order):
        if pd.notna(nes_mat.loc[pathway, ct]):
            ax.text(j, i, label_mat.loc[pathway, ct], ha="center", va="center", fontsize=7)

cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label("NES (SLC7A5 high vs low)")
plt.tight_layout()

fig_pdf = FIG_DIR / "Fig1E_Nucseq_SLC7A5_top20_bottom20_focused_GSEA_metabolism_mTOR_NFKB_heatmap.pdf"
fig_png = FIG_DIR / "Fig1E_Nucseq_SLC7A5_top20_bottom20_focused_GSEA_metabolism_mTOR_NFKB_heatmap.png"
plt.savefig(fig_pdf, bbox_inches="tight", dpi=300)
plt.savefig(fig_png, bbox_inches="tight", dpi=200)
plt.show()

print(f"Saved -> {fig_pdf}")
print(f"Saved -> {fig_png}")
print(f"DEG/GSEA tables -> {OUTDIR}")


In [ ]:
# Fig1E nucseq revised: Macrophage-only SLC7A5 top/bottom 20% DEG, GSEA, and PDF figures
import re
from pathlib import Path

import numpy as np
import pandas as pd
import scanpy as sc
import gseapy as gp
import matplotlib.pyplot as plt
from scipy import sparse

H5AD_PATH = artifact_path('50_Public_GEO_analysis/analysis/Data/Liver_DB_nuc_stage.h5ad')
FIG_DIR = Path(output_dir('02_SLC7A5_mono_Journal/05_Figure'))
OUTDIR = Path(output_dir('02_SLC7A5_mono_Journal/02_Analysis/Nucseq/Fig1E_Nucseq_Macrophage_SLC7A5_top20_bottom20_GSEA'))
FIG_DIR.mkdir(parents=True, exist_ok=True)
OUTDIR.mkdir(parents=True, exist_ok=True)

GENE = "SLC7A5"
CELLTYPE = "Macrophage"
TOP_BOTTOM_PCT = 0.20
MIN_CELLS_PER_GROUP = 25
LFC_CAP = 10.0
GROUP_COL = "SLC7A5_top20_bottom20_group"
EXCLUDE_STRATIFICATION_GENE_FROM_GSEA = True

if "adata" in globals():
    adata_nuc_my = adata.copy()
else:
    adata_nuc_my = sc.read_h5ad(H5AD_PATH)

print(adata_nuc_my)
print("raw exists:", adata_nuc_my.raw is not None)
print(adata_nuc_my.obs["celltype"].value_counts())

def get_gene_vector(adata_obj, gene):
    gene_upper = gene.upper()
    if adata_obj.raw is not None:
        raw_var_upper = pd.Index([str(g).upper() for g in adata_obj.raw.var_names])
        if gene_upper in raw_var_upper:
            raw_gene = adata_obj.raw.var_names[raw_var_upper.get_loc(gene_upper)]
            x = adata_obj.raw[:, raw_gene].X
            return np.asarray(x.toarray()).ravel() if sparse.issparse(x) else np.asarray(x).ravel()

    var_upper = pd.Index([str(g).upper() for g in adata_obj.var_names])
    if gene_upper not in var_upper:
        raise ValueError(f"{gene} was not found in adata.var_names or adata.raw.var_names")
    gene_name = adata_obj.var_names[var_upper.get_loc(gene_upper)]
    x = adata_obj[:, gene_name].X
    return np.asarray(x.toarray()).ravel() if sparse.issparse(x) else np.asarray(x).ravel()

adata_nuc_my.obs[f"{GENE}_expr_raw"] = get_gene_vector(adata_nuc_my, GENE)
sub = adata_nuc_my[adata_nuc_my.obs["celltype"].astype(str) == CELLTYPE].copy()

n_total = sub.n_obs
n_group = int(np.floor(n_total * TOP_BOTTOM_PCT))
if n_group < MIN_CELLS_PER_GROUP:
    raise ValueError(f"Not enough {CELLTYPE} cells for top/bottom split: n_total={n_total}, n_group={n_group}")

expr = sub.obs[f"{GENE}_expr_raw"].to_numpy()
order = np.argsort(expr, kind="mergesort")
group = np.full(n_total, "middle", dtype=object)
group[order[:n_group]] = "SLC7A5_low"
group[order[-n_group:]] = "SLC7A5_high"
sub.obs[GROUP_COL] = pd.Categorical(
    group,
    categories=["SLC7A5_low", "middle", "SLC7A5_high"],
    ordered=True,
)

sub_tb = sub[sub.obs[GROUP_COL] != "middle"].copy()
sub_tb.obs[GROUP_COL] = sub_tb.obs[GROUP_COL].cat.remove_unused_categories()

split_summary = pd.DataFrame([
    {
        "celltype": CELLTYPE,
        "total_cells": n_total,
        "n_per_group": n_group,
        "low_expr_min": float(expr[order[:n_group]].min()),
        "low_expr_max": float(expr[order[:n_group]].max()),
        "high_expr_min": float(expr[order[-n_group:]].min()),
        "high_expr_max": float(expr[order[-n_group:]].max()),
        "low_pct_positive": float((expr[order[:n_group]] > 0).mean()),
        "high_pct_positive": float((expr[order[-n_group:]] > 0).mean()),
    }
])
split_summary.to_csv(OUTDIR / "Nucseq_Macrophage_SLC7A5_top20_bottom20_split_summary.csv", index=False)
display(split_summary)

# DEG: nucseq Macrophage SLC7A5-high vs SLC7A5-low.
sc.tl.rank_genes_groups(
    sub_tb,
    groupby=GROUP_COL,
    groups=["SLC7A5_high"],
    reference="SLC7A5_low",
    method="wilcoxon",
    use_raw=sub_tb.raw is not None,
    pts=True,
)

deg_mac_nuc = sc.get.rank_genes_groups_df(sub_tb, group="SLC7A5_high").rename(columns={"names": "gene"})
for col in ["scores", "logfoldchanges", "pvals", "pvals_adj"]:
    deg_mac_nuc[col] = pd.to_numeric(deg_mac_nuc[col], errors="coerce")
deg_mac_nuc["logfoldchanges"] = deg_mac_nuc["logfoldchanges"].clip(-LFC_CAP, LFC_CAP)
deg_mac_nuc.insert(0, "celltype", CELLTYPE)

deg_path = OUTDIR / "DEG_Nucseq_Macrophage_SLC7A5_top20_high_vs_bottom20_low.csv"
deg_mac_nuc.to_csv(deg_path, index=False)
print(f"Saved DEG -> {deg_path}")
display(deg_mac_nuc.head(20))
display(deg_mac_nuc.loc[deg_mac_nuc["gene"].astype(str).str.upper().eq(GENE)])

# Preranked GSEA: Hallmark and KEGG. Focus on Glycolysis, OXPHOS, TCA cycle, FAO, NF-kB, TLR-LPS, and mTOR pathways.
GENE_SET_COLLECTIONS = {
    "Hallmark": "MSigDB_Hallmark_2020",
    "KEGG": "KEGG_2021_Human",
}

FOCUS_PATTERNS = {
    "Glycolysis": r"GLYCOLYSIS|GLUCONEOGENESIS",
    "OXPHOS": r"OXIDATIVE_PHOSPHORYLATION|RESPIRATORY_CHAIN|ELECTRON_TRANSPORT_CHAIN",
    "TCA cycle": r"TCA|CITRATE_CYCLE|TRICARBOXYLIC|CARBON_METABOLISM|PYRUVATE_METABOLISM",
    "FAO": r"FATTY_ACID|BETA_OXIDATION|ACYL_COA|PPAR_SIGNALING",
    "NF-kB": r"NFKB|NF_KB|TNFA_SIGNALING_VIA_NFKB",
    "TLR-LPS": r"TOLL_LIKE_RECEPTOR|LIPOPOLYSACCHARIDE|LPS|NOD_LIKE_RECEPTOR",
    "mTOR": r"MTOR|PI3K_AKT_MTOR|MTORC1",
}

PATHWAY_ORDER = ["Glycolysis", "OXPHOS", "TCA cycle", "FAO", "NF-kB", "TLR-LPS", "mTOR"]

def build_prerank(deg):
    rnk = deg[["gene", "logfoldchanges", "pvals", "scores"]].copy()
    rnk["gene"] = rnk["gene"].astype(str).str.upper()
    if EXCLUDE_STRATIFICATION_GENE_FROM_GSEA:
        rnk = rnk.loc[rnk["gene"] != GENE]
    rnk["pvals"] = pd.to_numeric(rnk["pvals"], errors="coerce").clip(lower=1e-300)
    rnk["logfoldchanges"] = pd.to_numeric(rnk["logfoldchanges"], errors="coerce")
    rnk["scores"] = pd.to_numeric(rnk["scores"], errors="coerce")
    rnk["rank_metric"] = np.sign(rnk["logfoldchanges"]) * -np.log10(rnk["pvals"])
    rnk.loc[rnk["rank_metric"].isna(), "rank_metric"] = rnk.loc[rnk["rank_metric"].isna(), "scores"]
    rnk = rnk.replace([np.inf, -np.inf], np.nan).dropna(subset=["gene", "rank_metric"])
    rnk = rnk.sort_values("rank_metric", ascending=False).drop_duplicates("gene", keep="first")
    return rnk[["gene", "rank_metric"]]

prerank_df = build_prerank(deg_mac_nuc)
rnk_path = OUTDIR / "Prerank_Nucseq_Macrophage_SLC7A5_top20_high_vs_low.rnk"
prerank_df.to_csv(rnk_path, sep="\t", index=False, header=False)
print(f"Saved prerank -> {rnk_path} | n_genes={len(prerank_df)}")

gsea_tables = []
for collection_label, gene_sets in GENE_SET_COLLECTIONS.items():
    try:
        pre_res = gp.prerank(
            rnk=prerank_df,
            gene_sets=gene_sets,
            min_size=5,
            max_size=500,
            permutation_num=1000,
            seed=7,
            outdir=None,
            verbose=False,
        )
    except Exception as e:
        print(f"[{collection_label}] failed with {gene_sets}: {e}")
        continue

    res = pre_res.res2d.copy()
    res.insert(0, "collection", collection_label)
    res.insert(1, "celltype", CELLTYPE)
    res_path = OUTDIR / f"GSEA_Nucseq_Macrophage_{collection_label}_SLC7A5_top20_high_vs_low.csv"
    res.to_csv(res_path, index=False)
    gsea_tables.append(res)
    print(f"[{collection_label}] saved -> {res_path} | terms={len(res)}")

if not gsea_tables:
    raise RuntimeError("No GSEA results were generated. Check gseapy gene set library access.")

gsea_all = pd.concat(gsea_tables, ignore_index=True)
gsea_all.to_csv(OUTDIR / "GSEA_Nucseq_Macrophage_all_collections_SLC7A5_top20_high_vs_low.csv", index=False)

focused = []
term_norm = gsea_all["Term"].astype(str).str.upper().str.replace(" ", "_", regex=False).str.replace("-", "_", regex=False)
for pathway_group, pattern in FOCUS_PATTERNS.items():
    hit = gsea_all.loc[term_norm.str.contains(pattern, regex=True, na=False)].copy()
    if hit.empty:
        print(f"No matched terms for {pathway_group}")
        continue
    hit.insert(2, "pathway_group", pathway_group)
    focused.append(hit)

focused_gsea_nuc = pd.concat(focused, ignore_index=True) if focused else pd.DataFrame()
if not focused_gsea_nuc.empty:
    focused_gsea_nuc["adjusted_pvalue"] = pd.to_numeric(focused_gsea_nuc["FDR q-val"], errors="coerce")
focused_gsea_nuc.to_csv(OUTDIR / "GSEA_Nucseq_Macrophage_focused_Glycolysis_OXPHOS_TCA_FAO_NFKB_TLRLPS_mTOR.csv", index=False)
print(f"Focused matched terms: {len(focused_gsea_nuc)}")
display(focused_gsea_nuc[["collection", "pathway_group", "Term", "NES", "NOM p-val", "adjusted_pvalue"]].head(50))

# Save focused GSEA PDF figures, matching Figure1.ipynb cell 58 style.
if focused_gsea_nuc.empty:
    raise ValueError("No focused GSEA terms matched. Inspect GSEA_Nucseq_Macrophage_all_collections_SLC7A5_top20_high_vs_low.csv manually.")

plot_df = focused_gsea_nuc.copy()
for col in ["NES", "NOM p-val", "adjusted_pvalue"]:
    plot_df[col] = pd.to_numeric(plot_df[col], errors="coerce")
plot_df["abs_NES"] = plot_df["NES"].abs()

best_df = (
    plot_df.sort_values(["pathway_group", "collection", "adjusted_pvalue", "abs_NES"], ascending=[True, True, True, False])
    .groupby(["pathway_group", "collection"], observed=True, as_index=False)
    .head(1)
    .copy()
)
best_df["pathway_group"] = pd.Categorical(best_df["pathway_group"], categories=PATHWAY_ORDER, ordered=True)
best_df = best_df.sort_values(["pathway_group", "collection"])
best_df["label"] = best_df["pathway_group"].astype(str) + " | " + best_df["collection"].astype(str)
best_df["term_short"] = best_df["Term"].astype(str).str.replace("HALLMARK_", "", regex=False).str.replace("KEGG_", "", regex=False)
best_df.to_csv(OUTDIR / "GSEA_Nucseq_Macrophage_focused_best_terms_for_pdf.csv", index=False)

fig_height = max(4.5, 0.42 * len(best_df) + 1.5)
fig, ax = plt.subplots(figsize=(8.5, fig_height))
colors = ["#b2182b" if v > 0 else "#2166ac" for v in best_df["NES"]]
ax.barh(best_df["label"], best_df["NES"], color=colors, alpha=0.88)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("NES (SLC7A5-high vs SLC7A5-low nucseq Macrophage)")
ax.set_ylabel("")
ax.set_title("Fig1E Nucseq. Macrophage SLC7A5-associated pathway enrichment")
ax.invert_yaxis()

x_min, x_max = ax.get_xlim()
x_span = x_max - x_min
for y, (_, row) in enumerate(best_df.iterrows()):
    star = "**" if row["adjusted_pvalue"] < 0.01 else "*" if row["adjusted_pvalue"] < 0.05 else ""
    txt = f"{star} adj p={row['adjusted_pvalue']:.3g}; {row['term_short']}"
    x = row["NES"] + (0.02 * x_span if row["NES"] >= 0 else -0.02 * x_span)
    ha = "left" if row["NES"] >= 0 else "right"
    ax.text(x, y, txt, va="center", ha=ha, fontsize=7)

plt.tight_layout()
bar_pdf = FIG_DIR / "Fig1E_Nucseq_Macrophage_SLC7A5_top20_bottom20_Hallmark_KEGG_focused_GSEA_barplot.pdf"
bar_png = FIG_DIR / "Fig1E_Nucseq_Macrophage_SLC7A5_top20_bottom20_Hallmark_KEGG_focused_GSEA_barplot.png"
plt.savefig(bar_pdf, bbox_inches="tight", dpi=300)
plt.savefig(bar_png, bbox_inches="tight", dpi=200)
plt.show()

# Dot plot with all focused matched terms, capped to the most significant terms for readability.
dot_df = plot_df.sort_values(["pathway_group", "adjusted_pvalue", "abs_NES"], ascending=[True, True, False]).groupby("pathway_group", observed=True).head(8).copy()
dot_df["pathway_group"] = pd.Categorical(dot_df["pathway_group"], categories=PATHWAY_ORDER, ordered=True)
dot_df = dot_df.sort_values(["pathway_group", "adjusted_pvalue", "abs_NES"], ascending=[True, True, False])
dot_df["minus_log10_adjusted_pvalue"] = -np.log10(dot_df["adjusted_pvalue"].clip(lower=1e-300))
dot_df["term_label"] = dot_df["collection"] + " | " + dot_df["Term"].astype(str).str.replace("HALLMARK_", "", regex=False).str.replace("KEGG_", "", regex=False)

fig_height = max(5, 0.28 * len(dot_df) + 1.6)
fig, ax = plt.subplots(figsize=(9.5, fig_height))
vlim = max(1, dot_df["NES"].abs().max())
scat = ax.scatter(
    dot_df["NES"],
    dot_df["term_label"],
    c=dot_df["NES"],
    s=20 + 28 * dot_df["minus_log10_adjusted_pvalue"].clip(upper=10),
    cmap="RdBu_r",
    vmin=-vlim,
    vmax=vlim,
    edgecolor="black",
    linewidth=0.25,
)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("NES (SLC7A5-high vs SLC7A5-low nucseq Macrophage)")
ax.set_ylabel("")
ax.set_title("Focused GSEA terms: Glycolysis, OXPHOS, TCA cycle, FAO, NF-kB, TLR-LPS, mTOR")
cbar = plt.colorbar(scat, ax=ax, pad=0.02)
cbar.set_label("NES")
plt.tight_layout()
dot_pdf = FIG_DIR / "Fig1E_Nucseq_Macrophage_SLC7A5_top20_bottom20_Hallmark_KEGG_focused_GSEA_dotplot.pdf"
dot_png = FIG_DIR / "Fig1E_Nucseq_Macrophage_SLC7A5_top20_bottom20_Hallmark_KEGG_focused_GSEA_dotplot.png"
plt.savefig(dot_pdf, bbox_inches="tight", dpi=300)
plt.savefig(dot_png, bbox_inches="tight", dpi=200)
plt.show()

print(f"Saved -> {bar_pdf}")
print(f"Saved -> {dot_pdf}")
print(f"Tables -> {OUTDIR}")


In [ ]:
# Supplementary Fig1X: nucseq macrophage SLC7A5-associated metabolic/starvation GSEA with adjusted p-values
# This cell directly visualizes the statement about TCA/TORC1, starvation responses, and amino-acid biosynthesis.
# It reuses the macrophage SLC7A5-high vs low DEG table generated above and adds GO BP terms.
import re
from pathlib import Path

import numpy as np
import pandas as pd
import gseapy as gp
import matplotlib.pyplot as plt
from gseapy.plot import gseaplot
from IPython.display import display

FIG_DIR = Path(output_dir('02_SLC7A5_mono_Journal/05_Figure'))
OUTDIR = Path(output_dir('02_SLC7A5_mono_Journal/02_Analysis/Nucseq/Fig1X_Nucseq_Macrophage_SLC7A5_metabolic_starvation_GSEA'))
DEG_PATH = Path(artifact_path('02_SLC7A5_mono_Journal/02_Analysis/Nucseq/Fig1E_Nucseq_Macrophage_SLC7A5_top20_bottom20_GSEA/DEG_Nucseq_Macrophage_SLC7A5_top20_high_vs_bottom20_low.csv'))
FIG_DIR.mkdir(parents=True, exist_ok=True)
OUTDIR.mkdir(parents=True, exist_ok=True)

GENE = "SLC7A5"
EXCLUDE_STRATIFICATION_GENE_FROM_GSEA = True

# Enrichr/MSigDB library names can differ slightly across gseapy releases.
GSEAPY_CACHE = Path.home() / ".cache" / "gseapy"
GENE_SET_COLLECTIONS = {
    "Hallmark": [
        GSEAPY_CACHE / "Enrichr.MSigDB_Hallmark_2020.gmt",
        "MSigDB_Hallmark_2020",
        "Hallmark_2020",
    ],
    "KEGG": [
        GSEAPY_CACHE / "Enrichr.KEGG_2021_Human.gmt",
        "KEGG_2021_Human",
        "KEGG_2019_Human",
    ],
    "GOBP": [
        GSEAPY_CACHE / "Enrichr.GO_Biological_Process_2023.gmt",
        "GO_Biological_Process_2023",
        "GO_Biological_Process_2021",
        "GO_Biological_Process_2018",
    ],
}
# Prefer local cached GMT files so the figure can be regenerated without Enrichr/MSigDB network access.
GENE_SET_COLLECTIONS = {
    label: [str(x) for x in libraries if not isinstance(x, Path) or x.exists()]
    for label, libraries in GENE_SET_COLLECTIONS.items()
}

FOCUS_PATTERNS = {
    "TCA cycle": r"TCA|CITRATE_CYCLE|TRICARBOXYLIC|PYRUVATE_METABOLISM|CARBON_METABOLISM",
    "TORC1/mTOR": r"TORC1|MTORC1|MTOR|PI3K_AKT_MTOR|MTOR_SIGNALING",
    "Amino acid starvation": r"AMINO_ACID_STARVATION|CELLULAR_RESPONSE_TO_AMINO_ACID_STARVATION|RESPONSE_TO_AMINO_ACID_STARVATION",
    "Glucose starvation": r"GLUCOSE_STARVATION|CELLULAR_RESPONSE_TO_GLUCOSE_STARVATION|RESPONSE_TO_GLUCOSE_STARVATION",
    "Starvation/nutrient response": r"STARVATION|NUTRIENT_LEVEL|NUTRIENT|NITROGEN_STARVATION",
    "Amino acid biosynthesis": r"AMINO_ACID.*BIOSYNTH|BIOSYNTH.*AMINO_ACID|ARGININE_BIOSYNTHESIS|AMINOACYL_TRNA_BIOSYNTHESIS|PHENYLALANINE.*TYROSINE.*TRYPTOPHAN_BIOSYNTHESIS|ALPHA_AMINO_ACID_BIOSYNTHETIC_PROCESS",
}
PATHWAY_ORDER = list(FOCUS_PATTERNS)


def build_prerank_from_deg(deg):
    rnk = deg[["gene", "logfoldchanges", "pvals", "scores"]].copy()
    rnk["gene"] = rnk["gene"].astype(str).str.upper()
    if EXCLUDE_STRATIFICATION_GENE_FROM_GSEA:
        rnk = rnk.loc[rnk["gene"] != GENE]
    rnk["pvals"] = pd.to_numeric(rnk["pvals"], errors="coerce").clip(lower=1e-300)
    rnk["logfoldchanges"] = pd.to_numeric(rnk["logfoldchanges"], errors="coerce")
    rnk["scores"] = pd.to_numeric(rnk["scores"], errors="coerce")
    rnk["rank_metric"] = np.sign(rnk["logfoldchanges"]) * -np.log10(rnk["pvals"])
    rnk.loc[rnk["rank_metric"].isna(), "rank_metric"] = rnk.loc[rnk["rank_metric"].isna(), "scores"]
    rnk = rnk.replace([np.inf, -np.inf], np.nan).dropna(subset=["gene", "rank_metric"])
    rnk = rnk.sort_values("rank_metric", ascending=False).drop_duplicates("gene", keep="first")
    return rnk[["gene", "rank_metric"]]


def normalize_term(term):
    return re.sub(r"[^A-Z0-9]+", "_", str(term).upper()).strip("_")


def run_prerank_with_fallback(rnk, label, libraries):
    last_error = None
    for gene_sets in libraries:
        try:
            pre_res = gp.prerank(
                rnk=rnk,
                gene_sets=gene_sets,
                processes=4,
                permutation_num=200,
                min_size=5,
                max_size=500,
                seed=7,
                outdir=None,
                verbose=False,
            )
            res = pre_res.res2d.copy()
            res.insert(0, "collection", label)
            res.insert(1, "gene_set_library", gene_sets)
            print(f"[{label}] GSEA complete with {gene_sets}: {res.shape[0]} terms")
            return pre_res, res
        except Exception as err:
            last_error = err
            print(f"[{label}] failed with {gene_sets}: {err}")
    print(f"[{label}] skipped. Last error: {last_error}")
    return None, pd.DataFrame()


def select_focused_terms(gsea_all):
    rows = []
    term_norm = gsea_all["Term"].map(normalize_term)
    for pathway_group, pattern in FOCUS_PATTERNS.items():
        hit = gsea_all.loc[term_norm.str.contains(pattern, regex=True, na=False)].copy()
        if hit.empty:
            continue
        hit.insert(2, "pathway_group", pathway_group)
        rows.append(hit)
    focused = pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()
    if focused.empty:
        return focused
    focused["NES"] = pd.to_numeric(focused["NES"], errors="coerce")
    focused["adjusted_pvalue"] = pd.to_numeric(focused["FDR q-val"], errors="coerce")
    focused["nominal_pvalue"] = pd.to_numeric(focused["NOM p-val"], errors="coerce")
    focused["direction"] = np.where(focused["NES"] >= 0, "SLC7A5-high", "SLC7A5-low")
    return focused


def choose_best_terms(focused):
    best_rows = []
    for pathway_group, sub in focused.groupby("pathway_group", observed=True):
        sub = sub.copy()
        sub["abs_nes"] = sub["NES"].abs()
        # Keep the strongest adjusted-p term per pathway concept; ties favor larger absolute NES.
        best = sub.sort_values(["adjusted_pvalue", "abs_nes"], ascending=[True, False]).iloc[0]
        best_rows.append(best)
    best = pd.DataFrame(best_rows)
    best["pathway_group"] = pd.Categorical(best["pathway_group"], categories=PATHWAY_ORDER, ordered=True)
    return best.sort_values("pathway_group")


def plot_focused_bar(best_df):
    plot_df = best_df.copy().sort_values("NES")
    fig_h = max(3.8, 0.45 * len(plot_df) + 1.2)
    fig, ax = plt.subplots(figsize=(8.2, fig_h))
    colors = np.where(plot_df["NES"] >= 0, "#d95f02", "#1b9e77")
    ax.barh(plot_df["pathway_group"].astype(str), plot_df["NES"], color=colors, alpha=0.88)
    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_xlabel("NES (SLC7A5-high vs SLC7A5-low nucseq macrophage)")
    ax.set_ylabel("")
    ax.set_title("SLC7A5-associated metabolic and starvation pathways")
    ax.grid(axis="x", color="#dddddd", linewidth=0.7)
    for y, (_, row) in enumerate(plot_df.iterrows()):
        adj = row["adjusted_pvalue"]
        label = f"adj. p={adj:.3g}" if pd.notna(adj) else "adj. p=NA"
        x = row["NES"]
        ha = "left" if x >= 0 else "right"
        offset = 0.03 if x >= 0 else -0.03
        ax.text(x + offset, y, label, va="center", ha=ha, fontsize=8)
    fig.tight_layout()
    pdf = FIG_DIR / "SupplFig1X_Nucseq_Macrophage_SLC7A5_metabolic_starvation_GSEA_barplot.pdf"
    png = FIG_DIR / "SupplFig1X_Nucseq_Macrophage_SLC7A5_metabolic_starvation_GSEA_barplot.png"
    fig.savefig(pdf, bbox_inches="tight")
    fig.savefig(png, dpi=300, bbox_inches="tight")
    plt.show()
    print(f"Saved barplot -> {pdf}")


def save_gseaplot(pre_res, term, pathway_group):
    safe_group = re.sub(r"[^A-Za-z0-9]+", "_", str(pathway_group)).strip("_")
    safe_term = re.sub(r"[^A-Za-z0-9]+", "_", str(term))[:90].strip("_")
    pdf = FIG_DIR / f"SupplFig1X_GSEAplot_{safe_group}_{safe_term}.pdf"
    png = FIG_DIR / f"SupplFig1X_GSEAplot_{safe_group}_{safe_term}.png"
    try:
        axes = pre_res.plot(terms=term, show_ranking=True, figsize=(5.4, 4.5))
        fig = axes[0].figure if isinstance(axes, list) else axes.figure
    except Exception:
        ax = gseaplot(rank_metric=pre_res.ranking, term=term, **pre_res.results[term])
        fig = ax[0].figure if isinstance(ax, list) else ax.figure
    fig.savefig(pdf, bbox_inches="tight")
    fig.savefig(png, dpi=300, bbox_inches="tight")
    plt.show()
    print(f"Saved GSEA plot -> {pdf}")


deg = pd.read_csv(DEG_PATH)
rnk = build_prerank_from_deg(deg)
rnk_path = OUTDIR / "Prerank_Nucseq_Macrophage_SLC7A5_top20_high_vs_low_for_Fig1X.rnk"
rnk.to_csv(rnk_path, sep="\t", index=False, header=False)
print(f"Ranked genes: {rnk.shape[0]} -> {rnk_path}")

pre_res_by_collection = {}
gsea_tables = []
for label, libraries in GENE_SET_COLLECTIONS.items():
    pre_res, res = run_prerank_with_fallback(rnk, label, libraries)
    if pre_res is not None:
        pre_res_by_collection[label] = pre_res
    if not res.empty:
        gsea_tables.append(res)

if not gsea_tables:
    raise RuntimeError("No GSEA result was generated. Check gseapy library access or internet connection.")

gsea_all = pd.concat(gsea_tables, ignore_index=True)
gsea_all["adjusted_pvalue"] = pd.to_numeric(gsea_all["FDR q-val"], errors="coerce")
gsea_all.to_csv(OUTDIR / "GSEA_Nucseq_Macrophage_metabolic_starvation_all_terms.csv", index=False)

focused = select_focused_terms(gsea_all)
focused.to_csv(OUTDIR / "GSEA_Nucseq_Macrophage_metabolic_starvation_focused_terms.csv", index=False)
if focused.empty:
    raise ValueError("No focused metabolic/starvation terms matched. Inspect all-term GSEA output manually.")

best = choose_best_terms(focused)
best.to_csv(OUTDIR / "GSEA_Nucseq_Macrophage_metabolic_starvation_best_terms_for_plot.csv", index=False)

display(best[["collection", "gene_set_library", "pathway_group", "Term", "NES", "nominal_pvalue", "adjusted_pvalue", "direction", "Lead_genes"]])
plot_focused_bar(best)

# One GSEA running-enrichment plot per selected pathway concept.
for _, row in best.iterrows():
    pre_res = pre_res_by_collection.get(row["collection"])
    if pre_res is None or row["Term"] not in pre_res.results:
        print(f"Skip GSEA plot: {row['Term']} not found in stored result object")
        continue
    save_gseaplot(pre_res, row["Term"], row["pathway_group"])

print(f"Tables -> {OUTDIR}")


In [ ]:
# Supplementary Fig1X add-on: ORA of SLC7A5-high up-DEGs and SLC7A5-low/down-DEGs with adjusted p-values
# This cell performs offline over-representation analysis using cached GO BP and KEGG GMT files.
# It is separate from the preranked GSEA above and supports statements that specifically mention ORA.

from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import hypergeom

FIG_DIR = Path(output_dir('02_SLC7A5_mono_Journal/05_Figure'))
OUTDIR = Path(output_dir('02_SLC7A5_mono_Journal/02_Analysis/Nucseq/Fig1X_Nucseq_Macrophage_SLC7A5_metabolic_starvation_ORA'))
DEG_PATH = Path(artifact_path('02_SLC7A5_mono_Journal/02_Analysis/Nucseq/Fig1E_Nucseq_Macrophage_SLC7A5_top20_bottom20_GSEA/DEG_Nucseq_Macrophage_SLC7A5_top20_high_vs_bottom20_low.csv'))
OUTDIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

PADJ_CUTOFF = 0.05
LOGFC_CUTOFF = 0.25
MIN_OVERLAP = 3
MIN_SET_SIZE = 5
MAX_SET_SIZE = 500
GENE = "SLC7A5"
EXCLUDE_STRATIFICATION_GENE_FROM_ORA = True

GSEAPY_CACHE = Path.home() / ".cache" / "gseapy"
GMT_LIBRARIES = {
    "KEGG": [
        GSEAPY_CACHE / "Enrichr.KEGG_2021_Human.gmt",
        GSEAPY_CACHE / "KEGG_2021_Human.gmt",
    ],
    "GOBP": [
        GSEAPY_CACHE / "Enrichr.GO_Biological_Process_2023.gmt",
        GSEAPY_CACHE / "GO_Biological_Process_2023.gmt",
        GSEAPY_CACHE / "Enrichr.GO_Biological_Process_2021.gmt",
        GSEAPY_CACHE / "GO_Biological_Process_2021.gmt",
    ],
}

FOCUSED_PATTERNS = {
    "TCA cycle": r"TCA|CITRATE_CYCLE|TRICARBOXYLIC|PYRUVATE_METABOLISM|CARBON_METABOLISM",
    "TORC1/mTOR": r"TORC1|MTORC1|MTOR|PI3K_AKT_MTOR|MTOR_SIGNALING",
    "Amino acid starvation": r"AMINO_ACID_STARVATION|CELLULAR_RESPONSE_TO_AMINO_ACID_STARVATION|RESPONSE_TO_AMINO_ACID_STARVATION",
    "Glucose starvation": r"GLUCOSE_STARVATION|CELLULAR_RESPONSE_TO_GLUCOSE_STARVATION|RESPONSE_TO_GLUCOSE_STARVATION",
    "Starvation/nutrient response": r"STARVATION|NUTRIENT_LEVEL|NITROGEN_STARVATION",
    "Amino acid biosynthesis": r"AMINO_ACID.*BIOSYNTH|BIOSYNTH.*AMINO_ACID|ARGININE_BIOSYNTHESIS|AMINOACYL_TRNA_BIOSYNTHESIS|PHENYLALANINE.*TYROSINE.*TRYPTOPHAN_BIOSYNTHESIS|ALPHA_AMINO_ACID_BIOSYNTHETIC_PROCESS",
}


def first_existing(paths):
    for path in paths:
        if Path(path).exists():
            return Path(path)
    raise FileNotFoundError("No cached GMT file found among: " + ", ".join(map(str, paths)))


def read_gmt(path):
    gene_sets = {}
    with open(path) as handle:
        for line in handle:
            parts = line.rstrip("\n").split("\t")
            if len(parts) < 3:
                continue
            term = parts[0]
            genes = {gene.upper() for gene in parts[2:] if gene}
            if genes:
                gene_sets[term] = genes
    return gene_sets


def bh_adjust(pvalues):
    pvalues = np.asarray(pvalues, dtype=float)
    adjusted = np.full(len(pvalues), np.nan)
    valid = np.isfinite(pvalues)
    if not valid.any():
        return adjusted
    p = pvalues[valid]
    order = np.argsort(p)
    ranked = p[order]
    n = len(ranked)
    q = ranked * n / np.arange(1, n + 1)
    q = np.minimum.accumulate(q[::-1])[::-1]
    q = np.clip(q, 0, 1)
    valid_indices = np.where(valid)[0]
    adjusted[valid_indices[order]] = q
    return adjusted


def run_ora(query_genes, background_genes, gene_sets, collection, direction):
    query = {gene.upper() for gene in query_genes if pd.notna(gene)} & background_genes
    background = set(background_genes)
    M = len(background)
    N = len(query)
    records = []

    if N == 0:
        return pd.DataFrame()

    for term, genes in gene_sets.items():
        term_genes = genes & background
        n = len(term_genes)
        if n < MIN_SET_SIZE or n > MAX_SET_SIZE:
            continue
        overlap = sorted(query & term_genes)
        k = len(overlap)
        if k < MIN_OVERLAP:
            continue
        pvalue = hypergeom.sf(k - 1, M, n, N)
        records.append({
            "collection": collection,
            "direction": direction,
            "Term": term,
            "query_size": N,
            "background_size": M,
            "term_size_in_background": n,
            "overlap_size": k,
            "overlap_ratio": k / N,
            "gene_ratio": f"{k}/{N}",
            "background_ratio": f"{n}/{M}",
            "pvalue": pvalue,
            "overlap_genes": ";".join(overlap),
        })

    res = pd.DataFrame(records)
    if res.empty:
        return res
    res["adjusted_pvalue"] = bh_adjust(res["pvalue"].to_numpy())
    res["minus_log10_adjusted_pvalue"] = -np.log10(res["adjusted_pvalue"].clip(lower=np.nextafter(0, 1)))
    return res.sort_values(["adjusted_pvalue", "pvalue", "overlap_size"], ascending=[True, True, False])


def assign_pathway_group(term):
    normalized = term.upper().replace(" ", "_").replace("-", "_")
    for group, pattern in FOCUSED_PATTERNS.items():
        if re.search(pattern, normalized):
            return group
    return np.nan


deg = pd.read_csv(DEG_PATH)
deg["gene_upper"] = deg["gene"].astype(str).str.upper()
deg = deg.dropna(subset=["gene", "logfoldchanges", "pvals_adj"])
deg = deg.drop_duplicates("gene_upper", keep="first")

if EXCLUDE_STRATIFICATION_GENE_FROM_ORA:
    deg = deg.loc[deg["gene_upper"] != GENE.upper()].copy()

background_genes = set(deg["gene_upper"])
high_up = deg.loc[
    (deg["pvals_adj"] < PADJ_CUTOFF) &
    (deg["logfoldchanges"] >= LOGFC_CUTOFF),
    "gene_upper"
]
low_down = deg.loc[
    (deg["pvals_adj"] < PADJ_CUTOFF) &
    (deg["logfoldchanges"] <= -LOGFC_CUTOFF),
    "gene_upper"
]

split_summary = pd.DataFrame([
    {"direction": "SLC7A5-high up-DEGs", "n_genes": len(high_up), "padj_cutoff": PADJ_CUTOFF, "logfc_cutoff": LOGFC_CUTOFF},
    {"direction": "SLC7A5-low/down-DEGs", "n_genes": len(low_down), "padj_cutoff": PADJ_CUTOFF, "logfc_cutoff": -LOGFC_CUTOFF},
    {"direction": "background", "n_genes": len(background_genes), "padj_cutoff": np.nan, "logfc_cutoff": np.nan},
])
split_summary.to_csv(OUTDIR / "ORA_Nucseq_Macrophage_SLC7A5_DEG_split_summary.csv", index=False)

ora_tables = []
loaded_libraries = []
for collection, paths in GMT_LIBRARIES.items():
    gmt_path = first_existing(paths)
    gene_sets = read_gmt(gmt_path)
    loaded_libraries.append({"collection": collection, "gmt_path": str(gmt_path), "n_gene_sets": len(gene_sets)})
    ora_tables.append(run_ora(high_up, background_genes, gene_sets, collection, "SLC7A5-high up-DEGs"))
    ora_tables.append(run_ora(low_down, background_genes, gene_sets, collection, "SLC7A5-low/down-DEGs"))

pd.DataFrame(loaded_libraries).to_csv(OUTDIR / "ORA_Nucseq_Macrophage_loaded_GMT_libraries.csv", index=False)
ora_all = pd.concat([tbl for tbl in ora_tables if not tbl.empty], ignore_index=True)
ora_all = ora_all.sort_values(["direction", "adjusted_pvalue", "pvalue"], ascending=[True, True, True])
ora_all.to_csv(OUTDIR / "ORA_Nucseq_Macrophage_SLC7A5_high_up_low_down_GO_KEGG_all_terms.csv", index=False)

ora_focused = ora_all.copy()
ora_focused["pathway_group"] = ora_focused["Term"].map(assign_pathway_group)
ora_focused = ora_focused.dropna(subset=["pathway_group"])
ora_focused = ora_focused.sort_values(["direction", "pathway_group", "adjusted_pvalue", "pvalue"])
ora_focused.to_csv(OUTDIR / "ORA_Nucseq_Macrophage_SLC7A5_high_up_low_down_GO_KEGG_focused_terms.csv", index=False)

# Compact best-term table for manuscript checking and plotting.
best = (
    ora_focused
    .sort_values(["direction", "pathway_group", "adjusted_pvalue", "pvalue"])
    .groupby(["direction", "pathway_group"], as_index=False)
    .head(1)
)
best.to_csv(OUTDIR / "ORA_Nucseq_Macrophage_SLC7A5_high_up_low_down_GO_KEGG_best_terms.csv", index=False)

if not best.empty:
    plot_df = best.copy()
    plot_df["label"] = plot_df["direction"] + " | " + plot_df["pathway_group"] + " | " + plot_df["collection"]
    plot_df = plot_df.sort_values("minus_log10_adjusted_pvalue")
    height = max(3, 0.35 * len(plot_df) + 1.2)
    fig, ax = plt.subplots(figsize=(8.5, height))
    colors = np.where(plot_df["direction"].eq("SLC7A5-high up-DEGs"), "#2f6f9f", "#b45f4d")
    ax.barh(plot_df["label"], plot_df["minus_log10_adjusted_pvalue"], color=colors)
    ax.axvline(-np.log10(0.05), color="black", linewidth=0.8, linestyle="--")
    ax.set_xlabel("-log10 adjusted p-value")
    ax.set_title("ORA of SLC7A5-associated macrophage DEGs")
    ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout()
    pdf = FIG_DIR / "SupplFig1X_Nucseq_Macrophage_SLC7A5_high_up_low_down_ORA_barplot.pdf"
    png = FIG_DIR / "SupplFig1X_Nucseq_Macrophage_SLC7A5_high_up_low_down_ORA_barplot.png"
    fig.savefig(pdf, bbox_inches="tight")
    fig.savefig(png, dpi=300, bbox_inches="tight")
    plt.show()
    print(f"Saved ORA barplot -> {pdf}")

print(f"DEG split summary -> {OUTDIR / 'ORA_Nucseq_Macrophage_SLC7A5_DEG_split_summary.csv'}")
print(f"ORA all terms -> {OUTDIR / 'ORA_Nucseq_Macrophage_SLC7A5_high_up_low_down_GO_KEGG_all_terms.csv'}")
print(f"ORA focused terms -> {OUTDIR / 'ORA_Nucseq_Macrophage_SLC7A5_high_up_low_down_GO_KEGG_focused_terms.csv'}")
print(f"ORA best terms -> {OUTDIR / 'ORA_Nucseq_Macrophage_SLC7A5_high_up_low_down_GO_KEGG_best_terms.csv'}")
display(split_summary)
display(best)
